In [ ]:
import pandas as pd
import requests
import os

In [ ]:
URL = "https://77.rosstat.gov.ru/storage/mediabank/Динамика%20денежных%20доходов%20населения%20г.%20Москвы%20за%202014-2024%20гг.(2).xlsx"


In [ ]:
# Скачиваем файл
r = requests.get(URL, verify=False)
r.raise_for_status()
with open("income.xlsx", "wb") as f:
    f.write(r.content)
print("Файл скачан")



In [ ]:
# Читаем Excel
df = pd.read_excel("income.xlsx", skiprows=2)  # пропуск строк с заголовками


In [ ]:
# Переименовываем первый столбец
df = df.rename(columns={df.columns[0]: "indicator"})


In [ ]:
# Приведение числовых столбцов к float
for col in df.columns[1:]:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(" ", "").str.replace(",", "."), errors="coerce")



In [ ]:
# Переводим в длинный формат
df_long = df.melt(id_vars="indicator", var_name="year", value_name="value")
df_long = df_long.dropna()


In [ ]:
# Сохраняем для дальнейшей работы
os.makedirs("data/processed", exist_ok=True)
df_long.to_csv("data/processed/income_cleaned.csv", index=False)
print("CSV сохранён")

In [ ]:
def preprocess_excel(file_path, skiprows=2):
    df = pd.read_excel(file_path, skiprows=skiprows)
    df = df.rename(columns={df.columns[0]: "indicator"})
    for col in df.columns[1:]:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(" ", "").str.replace(",", "."), errors="coerce")
    df_long = df.melt(id_vars="indicator", var_name="year", value_name="value")
    df_long = df_long.dropna()
    return df_long

income = preprocess_excel("income.xlsx")


In [ ]:
import pandas as pd

df = pd.read_csv("data/processed/income_cleaned.csv")
print(df.head())
print(df.info())

In [ ]:
import re

# Оставляем только цифры (год)
df['year'] = df['year'].astype(str).apply(lambda x: int(re.search(r'\d{4}', x).group()))

In [ ]:
df = pd.read_excel("income.xlsx", skiprows=2)
df = df.rename(columns={df.columns[0]: "indicator"})



In [ ]:
for col in df.columns[1:]:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(" ", "").str.replace(",", "."), errors="coerce")



In [ ]:
# Переводим в длинный формат
df_long = df.melt(id_vars="indicator", var_name="year", value_name="value")
df_long = df_long.dropna()



In [ ]:
# Чистка года и индикаторов
df_long['year'] = df_long['year'].astype(str).apply(lambda x: int(re.search(r'\d{4}', x).group()))
df_long['indicator'] = df_long['indicator'].str.replace('\n', ' ').str.strip()



In [ ]:

# Сохраняем CSV для базы данных
os.makedirs("data/db_ready", exist_ok=True)
df_long.to_csv("data/db_ready/income_for_db.csv", index=False)



In [ ]:
# Создание сводной таблицы для анализа
pivot = df_long.pivot(index='year', columns='indicator', values='value')



In [ ]:
# Чистим названия столбцов после pivot
pivot.columns = pivot.columns.str.replace('\n', ' ')  # переносы -> пробел
pivot.columns = pivot.columns.str.replace(r'\s+', ' ', regex=True)  # множественные пробелы -> один
pivot.columns = pivot.columns.str.replace(r'\d+\)$', '', regex=True)  # удаляем '3)', '2)' в конце
pivot.columns = pivot.columns.str.strip()  # убираем пробелы по краям

print(pivot.columns)  # проверяем, какие названия теперь

In [ ]:
# Пример графика: доходы и расходы в тыс.рублей
pivot[['Денежные доходы (тыс.рублей)', 'Потребительские расходы (тыс.рублей)']].plot(
    kind='bar', stacked=True, figsize=(12,6)
)
plt.title("Структура доходов и расходов населения Москвы")
plt.xlabel("Год")
plt.ylabel("тыс.рублей")
plt.grid(True)
plt.show()

In [ ]:
pivot[['Денежные доходы        (тыс.рублей)',
       'Потребительские расходы (тыс.рублей)']].plot(kind='bar', stacked=True, figsize=(12,6))
plt.title("Структура доходов и расходов населения Москвы")
plt.xlabel("Год")
plt.ylabel("тыс.рублей")
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(pivot.index, pivot['Денежные доходы        (тыс.рублей)'], label='Доходы')
plt.plot(pivot.index, pivot['Потребительские расходы (тыс.рублей)'], label='Расходы')
plt.title("Динамика доходов и расходов населения Москвы")
plt.xlabel("Год")
plt.ylabel("тыс.рублей")
plt.legend()
plt.grid(True)

# Добавление аннотаций
plt.annotate('Резкий рост 2022-2024', xy=(2023, 18e9), xytext=(2015, 20e9),
             arrowprops=dict(facecolor='black', shrink=0.05))
plt.show()